# 25 AG review page: data for the H3 review artifact

**Purpose.** To show the stage-1 and stage-2 results to coauthors who know ReEDS but not our methods, this
notebook collects the exports of notebooks 21, 22, 23, 24 and 26 into the data files of a multi-page review page
(plan of 2026-10-29). It computes no new ledger value. It reshapes the exports, takes medians and ranges,
and checks that the page can rebuild both resident views from their components.

**What it writes** (`results/exports/25/`):
- `review/ag_review.html` (a copy of `aglib/review/template.html`) and `review/data/core.json`,
  `review/data/stage2.json`, `review/data/markets.json` (notebook 23), `review/data/cendiv.json`
  (notebook 26): the page and its data;
- `fig_*.csv`: the reference-variant numbers behind each figure, so every plotted number traces to a file.

It opens no ReEDS outputs file and states no hypothesis.


In [1]:
import sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "nlpaper").is_dir():           # find z-ethan/ from wherever the kernel starts
    _p = _p.parent
sys.path.insert(0, str(_p))
import json
import re
import shutil
from itertools import product

import numpy as np
import pandas as pd

import nlpaper
from nlpaper import paths
from aglib import ledger as L
nlpaper.setup(threads=4)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)
ZE = _p
OUT = paths.exports("25")
PAGE = OUT / "review"
(PAGE / "data").mkdir(parents=True, exist_ok=True)
E21, E22, E24 = paths.exports("21"), paths.exports("22"), paths.exports("24")
TARGETS = ["eia", "aj", "iaea", "mck", "cop28", "eo"]
RATE_KEYS = list(L.RATES)                      # 1.5%, 2.0%, 2.5%, 3%, 7%, 5.36%
PV_COLS = [f"PV_{k}" for k in RATE_KEYS]
BASES = list(L.SCC_PAIRS)
HV = list(L.HEALTH)
MKEY = ["horizon", "support_form", "funding", "allowance_national", "allowance_state"]
REF = dict(horizon="A", support_form="lump", funding="T", allowance_national="free", allowance_state="free")
r3 = lambda x: float(round(float(x), 3))
mk = lambda row: "|".join(str(row[k]) for k in MKEY)

nlpaper setup: python 3.12.3 | numpy 1.26.4 | pandas 2.2.2 | matplotlib 3.8.4 | threads 4
determinism: seeds are explicit numpy Generators in each cell; results do not depend on PYTHONHASHSEED.


## 1. Inputs

The page uses only these exports. Actor values are summed over each actor's entries; the entry detail is
kept for stage 1.

In [2]:
AP1 = pd.read_csv(E21 / "ag_actor_pv.csv")
VW1 = pd.read_csv(E21 / "ag_views.csv")
AN1 = pd.read_csv(E21 / "ag_annual_A.csv")
EM1 = pd.read_csv(E21 / "ag_emissions_delta.csv")
CL1 = pd.read_csv(E21 / "ag_closure.csv")
RPS1 = pd.read_csv(E21 / "ag_closure_rps_terms.csv")
C4 = pd.read_csv(E21 / "ag_c4_support.csv")
C5 = pd.read_csv(E21 / "ag_c5_firstcut.csv")
C5N = pd.read_csv(E21 / "ag_c5_named_differences.csv")
BCHK = pd.read_csv(E22 / "ag_baseline_check.csv")
BLEV = pd.read_csv(E22 / "ag_baseline_levels.csv")
BAP = pd.read_csv(E22 / "ag_band_actor_pv.csv")
BVW = pd.read_csv(E22 / "ag_band_views.csv")
PAIRS = pd.read_csv(E24 / "ag2_pairs.csv")
AP2 = pd.read_parquet(E24 / "ag2_actor_pv.parquet")
VW2 = pd.read_parquet(E24 / "ag2_views.parquet")
CL2 = pd.read_parquet(E24 / "ag2_closure.parquet")
C32 = pd.read_csv(E24 / "ag2_c3.csv")
ALC = pd.read_csv(E24 / "ag2_allowance_checks.csv")
SCHED = pd.read_csv(paths.exports("07") / "Table1_deployment_schedules.csv")
AP1["allowance_national"] = AP1["allowance_national"].fillna("none")
VW1["allowance_national"] = VW1["allowance_national"].fillna("none")
for d in (AP1, VW1, AN1):
    d["allowance_national"] = d["allowance_national"].fillna("none")
# "environment" holds one entry per carbon basis and health variant, so its entries never sum; the page
# takes the environment from the view components instead
ACTORS = sorted((set(AP1["actor"]) | set(AP2["actor"])) - {"environment"})
CONDS = pd.read_csv(paths.exports("08") / "market_condition_switches.csv")
print(len(ACTORS), "actors;", AP2["case"].nunique(), "stage-2 runs;", len(PAIRS), "stage-2 pairs")

16 actors; 308 stage-2 runs; 696 stage-2 pairs


## 2. Per-run records

One record per policy run against its primary baseline: the 6 stage-1 runs and the 308 stage-2 runs.
Each record holds, for every money key (horizon, support form, funding, national and state allowance
treatment):
- the PV of every actor at every rate (2025 $B, PV to 2026);
- the components of the two resident views at each carbon-value basis.

The views are additive in their components (`aglib/ledger.py` `views`):
- View 1 = V1 money + (MCPF − 1) × tax base + GHG + health;
- View 2 = −Δ resource cost + (MCPF − 1) × tax base + GHG + health − foreign gain − φ × Π.

The tax base is read from the deadweight at MCPF 1.25. The cell asserts that the components rebuild
`view1` and `view2` on every row of both views tables.

In [3]:
def actor_block(ap):
    # {mkey: [[actor values in ACTORS order] per rate]}
    a = ap.groupby(MKEY + ["actor"])[PV_COLS].sum()
    out = {}
    for key, g in a.groupby(level=list(range(len(MKEY)))):
        g = g.droplevel(list(range(len(MKEY)))).reindex(ACTORS)
        out["|".join(map(str, key))] = [[None if np.isnan(v) else r3(v) for v in g[c].values] for c in PV_COLS]
    return out


def view_block(vw):
    # components per money key and basis, plus per-horizon parts; asserts the rebuild
    base = vw[(vw["mcpf"] == 1.25) & (vw["phi"] == 0.0) & (vw["health_model"] == HV[0])].copy()
    base["tax"] = base["deadweight"] / 0.25
    base["v1m"] = base["view1"] - base["deadweight"] - base["env_ghg"] - base["env_health"]
    base = base[MKEY + ["scc_basis", "v1m", "tax", "profits_domestic", "foreign_gain"]].rename(
        columns={"profits_domestic": "pi", "foreign_gain": "fo"})
    hpart = (vw.drop_duplicates(["horizon", "scc_basis", "health_model"])
             [["horizon", "scc_basis", "health_model", "minus_d_resource_cost", "env_ghg", "env_health"]]
             .rename(columns={"minus_d_resource_cost": "scres", "env_ghg": "ghg", "env_health": "hl"}))
    # the resource cost and GHG terms must not depend on the money key, nor health on anything but its variant
    chk = vw.groupby(["horizon", "scc_basis", "health_model"])[["minus_d_resource_cost", "env_ghg", "env_health"]].agg(np.ptp)
    assert float(chk.values.max()) < 1e-9
    # rebuild every row
    x = vw.merge(base, on=MKEY + ["scc_basis"], how="left").merge(hpart, on=["horizon", "scc_basis", "health_model"], how="left")
    assert len(x) == len(vw) and not x[["v1m", "scres"]].isna().any().any()
    dw = (x["mcpf"] - 1.0) * x["tax"]
    v1 = x["v1m"] + dw + x["ghg"] + x["hl"]
    v2 = x["scres"] + dw + x["ghg"] + x["hl"] - x["fo"] - x["phi"] * x["pi"]
    err = float(max((v1 - x["view1"]).abs().max(), (v2 - x["view2"]).abs().max()))
    assert err < 1e-6, err
    mo = {}
    for key, g in base.groupby(MKEY):
        g = g.set_index("scc_basis").reindex(BASES)
        mo["|".join(map(str, key))] = [[r3(v) for v in g.loc[b, ["v1m", "tax", "pi", "fo"]]] for b in BASES]
    ho = {}
    for hz, g in hpart.groupby("horizon"):
        g = g.set_index(["scc_basis", "health_model"])
        ho[hz] = dict(scres=[r3(g.loc[(b, HV[0]), "scres"]) for b in BASES],
                      ghg=[r3(g.loc[(b, HV[0]), "ghg"]) for b in BASES],
                      health=[[r3(g.loc[(b, v), "hl"]) for v in HV] for b in BASES])
    return mo, ho, err


RUNS, max_err = [], 0.0
for t in TARGETS:
    ap, vw = AP1[AP1["target"] == t], VW1[VW1["target"] == t]
    mo, ho, err = view_block(vw)
    max_err = max(max_err, err)
    RUNS.append(dict(stage="1", case=f"smr100_{t}_p50", target=t, condition="reference", baseline="fbR_none",
                     actors=actor_block(ap), vm=mo, vh=ho))
P2 = PAIRS[PAIRS["primary"]]
ap_g = dict(tuple(AP2[AP2["primary"]].groupby(["stage", "case"])))
vw_g = dict(tuple(VW2[VW2["primary"]].groupby(["stage", "case"])))
for _, p in P2.iterrows():
    mo, ho, err = view_block(vw_g[(p["stage"], p["case"])])
    max_err = max(max_err, err)
    RUNS.append(dict(stage=p["stage"], case=p["case"], target=p["target"], condition=p["condition"],
                     baseline=p["baseline"], actors=actor_block(ap_g[(p["stage"], p["case"])]), vm=mo, vh=ho))
print(len(RUNS), "run records; max rebuild error (2025 $B):", max_err)
assert len(RUNS) == 6 + 308

314 run records; max rebuild error (2025 $B): 1.0913936421275139e-11


**Spot check against the observation log.** The reference view variant (funding T, lump, horizon A,
global EPA 2023 2.0%, AP2 ACS, MCPF 1, φ = 0, allowances free) must give the stage-1 views in O09 and
the stage-2 medians in O20.

In [4]:
def view_of(run, key, basis="global EPA2023 2.0%", hv="AP2 ACS", mcpf=1.0, phi=0.0):
    # mirrors the page's JavaScript rebuild
    hz = key.split("|")[0]
    b = BASES.index(basis)
    v1m, tax, pi, fo = run["vm"][key][b]
    hp = run["vh"][hz]
    hl = hp["health"][b][HV.index(hv)]
    dw = (mcpf - 1) * tax
    return v1m + dw + hp["ghg"][b] + hl, hp["scres"][b] + dw + hp["ghg"][b] + hl - fo - phi * pi


def key_for(run):
    k = dict(REF)
    a = run["actors"]
    for alt in (("allowance_national", "none"), ("support_form", "none"), ("funding", "none")):
        if "|".join(k[x] for x in MKEY) not in a:
            k[alt[0]] = alt[1]
    key = "|".join(k[x] for x in MKEY)
    if key not in a:                               # credit runs: support and funding both "none"
        k.update(support_form="none", funding="none")
        key = "|".join(k[x] for x in MKEY)
    return key


rows = []
for run in RUNS:
    key = key_for(run)
    v1, v2 = view_of(run, key)
    rows.append(dict(stage=run["stage"], case=run["case"], target=run["target"], condition=run["condition"],
                     key=key, view1=v1, view2=v2))
REFV = pd.DataFrame(rows)
print(REFV[REFV["stage"] == "1"].round(1).to_string(index=False))
print(REFV[REFV["stage"].isin(["2a", "2c", "2d"])].groupby(["stage", "target"])[["view1", "view2"]]
      .median().unstack("stage").round(1).to_string())
REFV.to_csv(OUT / "fig_views_reference.csv", index=False)

stage             case target condition                key  view1  view2
    1   smr100_eia_p50    eia reference A|lump|T|none|free   41.7   -4.3
    1    smr100_aj_p50     aj reference A|lump|T|none|free   66.0   -0.3
    1  smr100_iaea_p50   iaea reference A|lump|T|none|free  206.4   34.2
    1   smr100_mck_p50    mck reference A|lump|T|none|free  313.0   98.1
    1 smr100_cop28_p50  cop28 reference A|lump|T|none|free  780.4  313.9
    1    smr100_eo_p50     eo reference A|lump|T|none|free 1427.6  709.9
         view1                 view2             
stage       2a     2c      2d     2a    2c     2d
target                                           
aj        78.1    7.5    53.7    6.5 -15.6    0.4
cop28    822.6  259.9  1437.2  362.5 -50.7  626.6
eia       48.9    5.4   145.1    0.1 -20.7   14.5
eo      1316.7  518.5  1557.3  575.8   5.8  679.4
iaea     266.9   68.7   861.6  106.1 -51.5  317.2
mck      296.1   94.7   191.7   79.3 -35.1   38.4


## 3. Stage-1 detail: entries, annual flows, emissions

In [5]:
ENTRIES = {}
for t, g in AP1.groupby("target"):
    ENTRIES[t] = {}
    for key, gg in g.groupby(MKEY):
        ENTRIES[t]["|".join(map(str, key))] = [[a, e, [r3(x) for x in vals]] for a, e, *vals in
                                               gg[["actor", "entry"] + PV_COLS].itertuples(index=False)
                                               if any(abs(x) > 5e-4 for x in vals)]
an = AN1[AN1["actor"] != "environment"].groupby(["target", "support_form", "funding", "allowance_state", "actor", "year"])["gain_2025B"].sum()
ANNUAL = {}
for (t, sf, fu, ast), g in an.groupby(level=[0, 1, 2, 3]):
    g = g.droplevel([0, 1, 2, 3]).unstack("year")
    ANNUAL.setdefault(t, {})[f"{sf}|{fu}|{ast}"] = {a: [r3(x) for x in g.loc[a].values] for a in g.index}
YEARS_A = sorted(AN1["year"].unique().tolist())
cum = EM1.groupby(["target", "entry"])["tonnes"].sum().unstack("entry") / 1e6
EMIS = dict(entries=list(cum.columns), cum_Mt={t: [r3(x) for x in cum.loc[t].values] for t in TARGETS},
            annual={t: {e: [r3(x) for x in g.sort_values("year")["tonnes"].values / 1e6]
                        for e, g in EM1[EM1["target"] == t].groupby("entry")} for t in TARGETS},
            years=sorted(EM1["year"].unique().tolist()))
cum.round(2).to_csv(OUT / "fig_emissions_cumulative_Mt.csv")
ref1 = AP1[(AP1["actor"] != "environment") & (AP1["horizon"] == "A") & (AP1["support_form"] == "lump") & (AP1["funding"] == "T")
           & (AP1["allowance_state"] == "free")].groupby(["actor", "target"])["PV_2.0%"].sum().unstack("target")[TARGETS]
ref1.round(2).to_csv(OUT / "fig_stage1_actors_reference.csv")
print(ref1.round(1).to_string())
print("emissions (Mt, cumulative 2026-2050):"); print(cum.round(1).to_string())

target                              eia    aj   iaea    mck  cop28     eo
actor                                                                    
electricity consumers              32.7  51.1  133.4  162.6  353.4  585.9
foreign buyers                      7.8   7.7   21.6   28.9   62.0   89.9
foreign sellers: Canadian imports  -1.2  -1.4   -3.1   -3.0   -4.9   -6.2
gas producers                     -26.9 -26.2  -74.6  -99.0 -206.7 -294.0
network                             0.6  -1.2   -5.4   -5.8  -12.9  -17.2
non-power gas consumers            10.8  10.4   29.4   39.3   84.3  121.9
owners: H2 and DAC production      -0.0  -0.5   -0.5   -0.2   -0.2   -0.1
owners: SMR (new)                   9.1  11.0   11.5   18.1   11.7   11.4
owners: coal                       -0.5  -0.2   -2.6   -2.6   -6.7  -10.2
owners: existing nuclear           -4.2  -5.4  -16.1  -19.2  -43.1  -70.4
owners: gas                        -3.3 -11.9  -19.3  -26.3  -68.3 -122.7
owners: hydro                      -3.

## 4. Closure checks

Stage 1 from notebook 21 (C1, C2, RPS by year; C3 by horizon and rate; C4; C5). Stage 2 from notebook 24:
the largest C1 and C2 share of gross per year over all pairs, and the largest C3 unexplained part.
The RPS trade wedge is split by origin from `extracts_ag/<batch>/derived_recs.parquet` (notebook 20 §6), the
one extract that this notebook reads directly. The split must sum to notebook 21's trade wedge.

In [6]:
yr = CL1[CL1["table"] == "C1/C2/RPS by year"]
CLOS1 = {}
for chk, g in yr.groupby("check"):
    CLOS1[chk] = {t: dict(t=gg["t"].astype(int).tolist(), share=[float(x) for x in gg["share_of_gross"]],
                          run=[r3(x) for x in gg["residual_run_B"]], base=[r3(x) for x in gg["residual_base_B"]],
                          delta=[r3(x) for x in gg["residual_delta_B"]])
                  for t, gg in g.groupby("target")}
c3 = CL1[CL1["check"] == "C3 views"]
C3T = [dict(target=r["target"], horizon=r["horizon"], rate=r["rate"], C3=r3(r["residual_max"]),
            C1=r3(r["term_C1 net"]), C2=r3(r["term_C2 net"]), RPS=r3(r["term_RPS"]),
            unexplained=float(r["unexplained"])) for _, r in c3.iterrows()]
RPST = {t: dict(t=g["t"].astype(int).tolist(), gap=[r3(x) for x in g["RPS_residual"]],
                trade_wedge=[r3(x) for x in g["trade_wedge"]],
                requirement_minus_RECs_bought=[r3(x) for x in g["requirement_minus_RECs_bought"]],
                minus_reporting_difference=[r3(x) for x in g["minus_reporting_difference"]])
        for t, g in RPS1.groupby("target")}
# fbR_none from notebook 24's market checks, which are in 2004 $ (notebook 21's terms are in 2004 $B)
b = CL2[(CL2["who"] == "base") & (CL2["baseline"] == "fbR_none")].drop_duplicates("t").sort_values("t")
RTERMS = ["RPS_residual", "trade_wedge", "requirement_minus_RECs_bought", "minus_reporting_difference"]
BR = pd.DataFrame({"RPS_residual": b["RPS_residual"].values, "trade_wedge": b["RPS_trade_rent"].values,
                   "requirement_minus_RECs_bought": b["RPS_requirement_minus_RECs_paid"].values,
                   "minus_reporting_difference": -b["RPS_revenue_minus_RECs_recv"].values},
                  index=b["t"].astype(int).values) / 1e9
RPST["fbR_none"] = dict(t=BR.index.tolist(), gap=[r3(x) for x in BR["RPS_residual"]],
                        **{k: [r3(x) for x in BR[k]] for k in RTERMS[1:]})
# units check: 2026 precedes every target's first mandated year, so run and baseline levels agree there
assert all(abs(RPST["fbR_none"]["gap"][0] - RPST[t]["gap"][0]) < 1e-3 for t in TARGETS), "fbR_none RPS units"
# PV of each delta RPS term (run - fbR_none) at 2.0%, by the ledger's flow rule; the gap must equal notebook 21's C3 term
RPSPV, rows = {}, []
for t, g in RPS1.groupby("target"):
    dlt = g.set_index("t")[RTERMS] - BR.reindex(g["t"].astype(int).values)[RTERMS].values
    assert not dlt.isna().any().any(), t
    RPSPV[t] = {}
    for hz in ("A", "B"):
        pvs = {k: L.pv(L.flow(dlt[k] * 1e9, hz), hz, L.RATES["2.0%"]) for k in RTERMS}
        ref = float(c3.loc[(c3["target"] == t) & (c3["horizon"] == hz) & (c3["rate"] == "2.0%"), "term_RPS"].iloc[0])
        assert abs(pvs["RPS_residual"] - ref) < 1e-6, (t, hz, pvs["RPS_residual"], ref)
        RPSPV[t][hz] = {("gap" if k == "RPS_residual" else k): r3(v) for k, v in pvs.items()}
        rows.append(dict(target=t, horizon=hz, rate="2.0%", **pvs))
pd.DataFrame(rows).to_csv(OUT / "fig_rps_delta_pv.csv", index=False)
print("delta RPS gap PV equals the notebook-21 C3 term in all 12 target-horizon cells")
# the trade wedge by origin (notebook 20 §6, aglib.reduce.reduce_run_recs): levels in 2004 $B and REC TWh per
# year, and the PV of each origin's change at 2.0%; the origins must sum to the trade wedge of notebook 21
WORIG = ["seller_unbound", "double_count", "trade_limits", "voluntary"]
QORIG = ["seller_unbound", "seller_priced", "voluntary"]
def recs_of(b, c, years):
    d = pd.read_parquet(paths.RESULTS / "extracts_ag" / b / "derived_recs.parquet")
    d = d[d["case"] == c].pivot_table(index="t", columns="item", values="Value", aggfunc="sum")
    return d.reindex(index=years, columns=[f"wedge|{o}" for o in WORIG] + [f"Q|{o}" for o in QORIG]).fillna(0.0)
SRC = {t: ("smr100", f"smr100_{t}_p50") for t in TARGETS} | {"fbR_none": ("itcfb", "fbR_none")}
RW = {k: recs_of(*bc, RPST[k]["t"]) for k, bc in SRC.items()}
for k, (b_, c_) in SRC.items():                       # smallest buyer/seller price ratio, RECs from priced sellers
    d = pd.read_parquet(paths.RESULTS / "extracts_ag" / b_ / "derived_recs.parquet")
    RPST[k]["min_ratio"] = round(float(d[(d["case"] == c_) & (d["item"] == "min_ratio|seller_priced") & (d["t"] >= 2026)]["Value"].min()), 4)
print({k: RPST[k]["min_ratio"] for k in SRC})
for k, d in RW.items():
    tw = np.array(RPST[k]["trade_wedge"])
    err = np.abs(d[[f"wedge|{o}" for o in WORIG]].sum(axis=1).values / 1e9 - tw).max()
    assert err < 1e-3, (k, err)                        # trade_wedge is rounded to 0.001
    RPST[k].update({f"w_{o}": [r3(x / 1e9) for x in d[f"wedge|{o}"]] for o in WORIG})
    RPST[k].update({f"q_{o}": [r3(x / 1e6) for x in d[f"Q|{o}"]] for o in QORIG})
wrows = []
for t in TARGETS:
    dlt = RW[t] - RW["fbR_none"].reindex(RW[t].index).values
    for hz in ("A", "B"):
        pvs = {o: L.pv(L.flow(dlt[f"wedge|{o}"], hz), hz, L.RATES["2.0%"]) for o in WORIG}
        assert abs(sum(pvs.values()) - RPSPV[t][hz]["trade_wedge"]) < 2e-3, (t, hz)
        RPSPV[t][hz].update({f"w_{o}": r3(v) for o, v in pvs.items()})
        wrows.append(dict(target=t, horizon=hz, rate="2.0%", **pvs))
pd.concat([d.assign(run=k).reset_index().rename(columns={"index": "t"}) for k, d in RW.items()]).to_csv(OUT / "fig_rps_wedge_origin.csv", index=False)
pd.DataFrame(wrows).to_csv(OUT / "fig_rps_wedge_origin_delta_pv.csv", index=False)
print("wedge origins sum to the notebook-21 trade wedge in all seven runs and to its delta PV in all 12 cells")
x = CL2.copy()
for c in ("C1", "C2"):
    x[f"{c}_share"] = (x[f"{c}_residual"] - x[f"{c}_vsc_rent"]).abs() / x[f"{c}_gross"].abs()
s2 = x.groupby(["stage", "t"])[["C1_share", "C2_share"]].max().reset_index()
CLOS2 = {st: dict(t=g["t"].astype(int).tolist(), C1=[float(v) for v in g["C1_share"]], C2=[float(v) for v in g["C2_share"]])
         for st, g in s2.groupby("stage")}
C3S2 = C32.groupby("stage").agg(pairs=("case", "size"), max_abs_unexplained=("unexplained", lambda s: float(s.abs().max())),
                                 max_spread=("C3_spread", lambda s: float(s.abs().max()))).reset_index()
ALS = ALC.groupby("stage").agg(pairs=("case", "size"),
                               emit_irt_minus_emit_nat_Mt=("emit_irt_minus_emit_nat_Mt", lambda s: float(s.abs().max())),
                               state_import_part_B_min=("state_import_part_B_min", "min"),
                               rggi_slack_B_maxabs=("rggi_slack_B_maxabs", lambda s: float(s.abs().max())),
                               co2_price_max=("co2_price_max", "max")).reset_index()
s2.to_csv(OUT / "fig_closure_stage2_share_by_year.csv", index=False)
print(C3S2.to_string(index=False)); print(ALS.to_string(index=False))
print("stage-2 max C1/C2 share:", float(s2["C1_share"].max()), float(s2["C2_share"].max()))
C4S = dict(n=int(len(C4)), max_rel=float(C4["rel_diff"].abs().max()))
C5S = dict(n=int(len(C5)), max_rel=float(C5["rel_diff"].abs().max()),
           named=[{k: (r3(v) if isinstance(v, float) else v) for k, v in r.items()} for r in C5N.to_dict("records")])
print(C4S, {k: v for k, v in C5S.items() if k != "named"})

delta RPS gap PV equals the notebook-21 C3 term in all 12 target-horizon cells
{'eia': 2.0, 'aj': 2.0, 'iaea': 1.9951, 'mck': 1.9951, 'cop28': 1.9926, 'eo': 1.9931, 'fbR_none': 2.0}
wedge origins sum to the notebook-21 trade wedge in all seven runs and to its delta PV in all 12 cells
stage  pairs  max_abs_unexplained  max_spread
   2a    600             0.000669    0.000005
   2b   3560             0.000972    0.000006
   2c   2400             0.000949    0.000071
   2d    400             0.000456    0.000005
stage  pairs  emit_irt_minus_emit_nat_Mt  state_import_part_B_min  rggi_slack_B_maxabs  co2_price_max
   2a     60                    0.000064                      0.0         5.490884e-09       0.000000
   2b    356                    0.000128                      0.0         1.481044e-08       0.000000
   2c    240                    0.000064                      0.0         2.144133e-09     152.620087
   2d     40                    0.000064                      0.0         3.0

## 5. Baselines and the bands

- The 19 baseline checks and their levels, from notebook 22.
- The reference band: the max − min of each actor and of each view across the 10 reference baselines
  without a nuclear ITC (A13), reference display. `fbR_itc30` is reported beside it, as its difference
  from `fbR_none`.
- The cap band: the same width across the 8 cap baselines, per stage-2c run; the page shows its largest value per target.

In [7]:
BCHK_R = BCHK.to_dict("records")
LEV = {}
for v in ["energy_price_per_MWh", "P_E_B", "P_C_B", "P_R_B", "systemcost_B", "gas_price_per_MMBtu", "CO2_process_Mt"]:
    LEV[v] = {c: dict(regime=g["regime"].iloc[0], t=g["t"].astype(int).tolist(), y=[r3(x) for x in g[v]])
              for c, g in BLEV.sort_values("t").groupby("case")}
refsel = lambda d: d[(d["support_form"] == "lump") & (d["funding"] == "T") & (d["allowance_state"] == "free")]
ba = refsel(BAP).groupby(["horizon", "target", "baseline", "in_band", "actor"])[["PV_2.0%", "PV_5.36%"]].sum().reset_index()
band = ba[ba["in_band"]].groupby(["horizon", "target", "actor"])[["PV_2.0%", "PV_5.36%"]].agg(lambda s: s.max() - s.min())
itc = (ba[ba["baseline"] == "fbR_itc30"].set_index(["horizon", "target", "actor"])[["PV_2.0%", "PV_5.36%"]]
       - ba[ba["baseline"] == "fbR_none"].set_index(["horizon", "target", "actor"])[["PV_2.0%", "PV_5.36%"]])
BAND = {}
for (hz, t), g in band.groupby(level=[0, 1]):
    g = g.droplevel([0, 1]).reindex(ACTORS)
    i = itc.loc[(hz, t)].reindex(ACTORS)
    BAND.setdefault(hz, {})[t] = dict(width_2=[None if np.isnan(v) else r3(v) for v in g["PV_2.0%"]],
                                      width_536=[None if np.isnan(v) else r3(v) for v in g["PV_5.36%"]],
                                      itc30_2=[None if np.isnan(v) else r3(v) for v in i["PV_2.0%"]])
bv = BVW[(BVW["support_form"] == "lump") & (BVW["funding"] == "T") & (BVW["allowance_state"] == "free")
         & (BVW["scc_basis"] == "global EPA2023 2.0%") & (BVW["health_model"] == "AP2 ACS")
         & (BVW["mcpf"] == 1.0) & (BVW["phi"] == 0.0)]
bvw = bv[bv["in_band"]].groupby(["horizon", "target"])[["view1", "view2"]].agg(lambda s: s.max() - s.min())
bvi = (bv[bv["baseline"] == "fbR_itc30"].set_index(["horizon", "target"])[["view1", "view2"]]
       - bv[bv["baseline"] == "fbR_none"].set_index(["horizon", "target"])[["view1", "view2"]])
BANDV = {hz: {t: dict(width=[r3(bvw.loc[(hz, t), "view1"]), r3(bvw.loc[(hz, t), "view2"])],
                      itc30=[r3(bvi.loc[(hz, t), "view1"]), r3(bvi.loc[(hz, t), "view2"])]) for t in TARGETS}
         for hz in ("A", "B")}
band.round(3).to_csv(OUT / "fig_band_reference_width.csv")
# cap band (stage 2c, all 8 cap baselines)
capsel = AP2[(AP2["stage"] == "2c") & (AP2["support_form"] == "lump") & (AP2["funding"] == "T")
             & (AP2["allowance_state"] == "free") & (AP2["allowance_national"] == "free")]
cw = (capsel.groupby(["horizon", "case", "target", "baseline", "actor"])["PV_2.0%"].sum()
      .groupby(["horizon", "case", "target", "actor"]).agg(lambda s: s.max() - s.min())
      .groupby(["horizon", "target", "actor"]).max())
cv = VW2[(VW2["stage"] == "2c") & (VW2["support_form"] == "lump") & (VW2["funding"] == "T")
         & (VW2["allowance_state"] == "free") & (VW2["allowance_national"] == "free")
         & (VW2["scc_basis"] == "global EPA2023 2.0%") & (VW2["health_model"] == "AP2 ACS")
         & (VW2["mcpf"] == 1.0) & (VW2["phi"] == 0.0)]
cvw = (cv.groupby(["horizon", "case", "target"])[["view1", "view2"]].agg(lambda s: s.max() - s.min())
       .groupby(["horizon", "target"]).max())
CAPBAND = {hz: {t: dict(actors=[None if np.isnan(v) else r3(v) for v in cw.loc[(hz, t)].reindex(ACTORS).values],
                        views=[r3(cvw.loc[(hz, t), "view1"]), r3(cvw.loc[(hz, t), "view2"])]) for t in TARGETS}
           for hz in ("A", "B")}
cw.round(3).to_csv(OUT / "fig_band_cap_width.csv")
print("reference band, max width over actors (A, 2.0%):", float(band.loc["A"]["PV_2.0%"].max()))
print("cap band views (A):"); print(cvw.loc["A"].round(3).to_string())

reference band, max width over actors (A, 2.0%): 0.8359715964445371
cap band views (A):
        view1  view2
target              
aj      0.009  0.313
cop28   0.009  0.313
eia     0.009  0.313
eo      0.009  0.313
iaea    0.009  0.313
mck     0.009  0.313


## 5b. RUN-3: the runs without the REC export double count (notebook 28)

Notebook 28 compares the RUN-3 fix runs and controls with their archived source runs. This cell copies its
exports into `core.json` as `run3`: the reproduction check (controls), the trade wedge by origin and the REC
quantities (2004 $B and TWh, summed 2026–2050), RPS payments, CO2 and 2050 capacity, the stage-1 ledger
(actor sums, horizon A, 2.0%, funding T, lump, allowances free) and the reference views from the source runs and
from the fix runs. The source-pair values must equal notebook 21's.

In [8]:
E28 = paths.exports("28")
R3 = {n: pd.read_csv(E28 / f"run3_{n}.csv") for n in
      ["reproduction", "control_ledger", "recs_by_origin", "other", "cap2050", "fix_ledger", "pair_actor", "pair_views", "rps_account"]}
NAMES3 = ["fbR_none"] + TARGETS
ro = R3["recs_by_origin"].set_index(["who", "run"])
WCOL = ["wedge|seller_unbound", "wedge|double_count", "wedge|trade_limits", "wedge|total", "Q|seller_unbound", "Q|seller_priced"]
rec3 = {w: {k: {c.replace("|", "_"): r3(ro.loc[(w, k), c]) for c in WCOL} for k in (NAMES3 if w != "control" else ["fbR_none", "eo"])}
        for w in ("source", "fix", "control")}
rep = R3["reproduction"]
cl = R3["control_ledger"].set_index("actor")
fl = R3["fix_ledger"].set_index("actor")
repro = dict(max_pct={k: r3(float(g["control - source, %"].abs().max())) for k, g in rep.groupby("run")},
             max_pct_money={k: r3(float(g.loc[~g["quantity"].str.startswith("recs|Q|"), "control - source, %"].abs().max()))
                            for k, g in rep.groupby("run")},
             q_unbound_pct={k: r3(float(g.loc[g["quantity"] == "recs|Q|seller_unbound", "control - source, %"].iloc[0]))
                            for k, g in rep.groupby("run")},
             ledger_max={k: r3(float(cl[k].abs().max())) for k in cl.columns},
             fix_ledger_max={k: r3(float(fl[k].abs().max())) for k in ["fbR_none", "eo"]})
oth = R3["other"].set_index(["run", "quantity"])
OKEY = {"ledger|P_R": "P_R", "system cost (no credit rows)": "syscost", "CO2 process, Mt": "co2", "support_mean, 2004 $/MW": "support"}
other3 = {k: {ok: dict(source=r3(oth.loc[(k, q), "source"]), fix=r3(oth.loc[(k, q), "fix"]), pct=r3(oth.loc[(k, q), "%"]))
              for q, ok in OKEY.items()} for k in NAMES3}
cap3 = R3["cap2050"].set_index("group")[NAMES3]
pa = R3["pair_actor"].set_index(["who", "actor"])
actors3 = {w: {a: [r3(pa.loc[(w, a), t]) for t in TARGETS] for a in pa.loc[w].index} for w in ("source", "fix")}
# the source pairs must equal notebook 21's actor sums (funding T, lump, horizon A, 2.0%, allowances free)
ap21 = AP1[(AP1["funding"] == "T") & (AP1["support_form"] == "lump") & (AP1["horizon"] == "A")
           & AP1["allowance_state"].isin(["free", "none"]) & AP1["allowance_national"].isin(["free", "none"])]
ap21 = ap21.groupby(["actor", "target"])["PV_2.0%"].sum().unstack("target")[TARGETS]
d21 = (pa.loc["source"][TARGETS] - ap21.reindex(pa.loc["source"].index)).abs().max().max()
assert d21 < 1e-6, d21
pv3 = R3["pair_views"].set_index(["who", "target", "horizon"])
views3 = {w: {hz: {c: [r3(pv3.loc[(w, t, hz), c]) for t in TARGETS] for c in ["view1", "view2", "profits_domestic"]}
              for hz in ("A", "B")} for w in ("source", "fix")}
ra = R3["rps_account"]
rpsacc3 = {w: {c: [r3(ra[(ra["who"] == w) & (ra["target"] == t) & (ra["run"] == "run")][c].iloc[0]) for t in TARGETS]
               + [r3(ra[(ra["who"] == w) & (ra["target"] == "eo") & (ra["run"] == "base")][c].iloc[0])]
               for c in ["RPS_residual", "trade_wedge", "RPS_gross"]} for w in ("source", "fix")}
SRC3 = {"fbR_none": ("itcfb", "fbR_none"), **{t: ("smr100", f"smr100_{t}_p50") for t in TARGETS}}
FIX3 = {"fbR_none": ("ag", "recfix_fbR_none"), **{t: ("ag", f"recfix_smr100_{t}_p50") for t in TARGETS}}
def min_ratio(b, c):
    d = pd.read_parquet(paths.RESULTS / "extracts_ag" / b / "derived_recs.parquet")
    return r3(float(d[(d["case"] == c) & (d["item"] == "min_ratio|seller_priced") & (d["t"] >= 2026) & (d["t"] <= 2050)]["Value"].min()))
MR3 = {w: {k: min_ratio(*bc) for k, bc in D.items()} for w, D in (("source", SRC3), ("fix", FIX3))}
print("smallest buyer/seller price ratio, RECs from priced sellers:", MR3)
RUN3 = dict(names=NAMES3, recs=rec3, repro=repro, other=other3,
            cap2050={g: [r3(v) for v in cap3.loc[g]] for g in cap3.index if (cap3.loc[g].abs() > 0.05).any()},
            actors=actors3, views=views3, rps_account=rpsacc3,
            min_ratio=MR3)
print("source pairs equal notebook 21 to", f"{d21:.1e}")
print("reproduction:", repro)


smallest buyer/seller price ratio, RECs from priced sellers: {'source': {'fbR_none': 2.0, 'eia': 2.0, 'aj': 2.0, 'iaea': 1.995, 'mck': 1.995, 'cop28': 1.993, 'eo': 1.993}, 'fix': {'fbR_none': 1.0, 'eia': 1.0, 'aj': 1.0, 'iaea': 1.0, 'mck': 1.0, 'cop28': 1.0, 'eo': 0.944}}
source pairs equal notebook 21 to 1.1e-13
reproduction: {'max_pct': {'eo': 4.002, 'fbR_none': 1.964}, 'max_pct_money': {'eo': 0.027, 'fbR_none': 0.058}, 'q_unbound_pct': {'eo': 4.002, 'fbR_none': -1.964}, 'ledger_max': {'fbR_none': 0.045, 'eo': 0.483}, 'fix_ledger_max': {'fbR_none': 15.994, 'eo': 19.676}}


## 6. Observation log, constants, and the page files

In [9]:
OBS = []
for line in (ZE / "docs" / "observations.md").read_text(encoding="utf-8").splitlines():
    m = re.match(r"^\| (O\d\d) \| ([^|]+) \| (.+) \| (.+) \| ([^|]+) \|$", line)
    if m:
        OBS.append(dict(id=m.group(1), date=m.group(2).strip(), source=m.group(3).strip(),
                        seen=m.group(4).strip(), status=m.group(5).strip()))
print(len(OBS), "observations")
CONST = dict(to2025=float(L.TO2025_FROM_2004), pvf=float(L.PVF_PAPER1), dev_rate=float(L.DEV_RATE),
             crf=float(1.0 / L.PVF_PAPER1), annuity_years=int(L.ANNUITY_YEARS),
             mcpf=L.MCPF, phi=L.PHI, rates=RATE_KEYS, bases=BASES, scc_rate=L.SCC_PAIRS, health=HV,
             years_A=[int(L.Y_A[0]), int(L.Y_A[-1])], years_B=[int(L.Y_B[0]), int(L.Y_B[-1])],
             ca_utility_share=float(L.CA_UTILITY_SHARE))
counts = PAIRS.groupby(["stage", "target"]).agg(runs=("case", "nunique"), pairs=("case", "size")).reset_index()
SCHED_R = SCHED.to_dict("records")
core = dict(built="2026-10-30", actors=ACTORS, targets=TARGETS, mkey=MKEY, ref=REF, const=CONST,
            runs=[r for r in RUNS if r["stage"] == "1"], entries=ENTRIES, annual=dict(years=YEARS_A, data=ANNUAL),
            emissions=EMIS, closure1=CLOS1, c3=C3T, rps=RPST, rps_pv=RPSPV, closure2=CLOS2, c3_stage2=C3S2.to_dict("records"),
            allowance_checks=ALS.to_dict("records"), c4=C4S, c5=C5S, baseline_check=BCHK_R, levels=LEV,
            band=BAND, band_views=BANDV, cap_band=CAPBAND, counts=counts.to_dict("records"),
            schedules=SCHED_R, observations=OBS,
            conditions=CONDS.groupby(["condition", "label"])["switch"].apply(list).reset_index().to_dict("records"),
            run3=RUN3,
            pair_totals=PAIRS.groupby("stage").agg(runs=("case", "nunique"), pairs=("case", "size")).reset_index().to_dict("records"))
stage2 = dict(runs=[r for r in RUNS if r["stage"] != "1"])
clean = lambda o: json.loads(json.dumps(o, default=lambda v: v.item() if hasattr(v, "item") else str(v)))
for name, obj in (("core", core), ("stage2", stage2)):
    s = json.dumps(clean(obj), separators=(",", ":"), allow_nan=False)
    (PAGE / "data" / f"{name}.json").write_text(s, encoding="utf-8")
    print(f"{name}.json: {len(s) / 1e6:.2f} MB")
shutil.copyfile(ZE / "aglib" / "review" / "template.html", PAGE / "ag_review.html")
print("page:", PAGE / "ag_review.html")

40 observations
core.json: 1.67 MB


stage2.json: 7.12 MB
page: C:\Users\ethan\code\research\ReEDS-paper-AG\z-ethan\results\exports\25\review\ag_review.html


## 7. Markets and census divisions (notebooks 23 and 26)

The markets page shows the 2050 differences and the 2026–2050 builds by technology group, for each run
set, as the median, minimum, maximum and number of runs per target. The census-division page shows each
division's PV for every horizon and actor rate, in the same form. Both checks below compare the page
values of stage 1 with the tables that notebooks 23 and 26 wrote.

In [10]:
E23, E26 = paths.exports("23"), paths.exports("26")
GROUPS = list(L.GROUPS)
Y1 = 2050
SET_LABEL = {"1|reference": "Stage 1: P50 runs", "2a|reference": "Stage 2a: cost samples",
             "2c|co2cap": "Stage 2c: national CO2 cap (primary cap baseline)", "2d|reference": "Stage 2d: credit runs"}
for c in ["gaslo", "gashi", "demhi", "relo", "rehi", "translim"]:
    SET_LABEL[f"2b|{c}"] = f"Stage 2b: {c} (programme, in the condition)"
SETS = list(SET_LABEL)
setkey = lambda d: d["stage"].astype(str) + "|" + d["condition"].astype(str)
r4 = lambda x: None if x is None or not np.isfinite(x) else float(round(float(x), 4))


def stats(d, by, col, scale=1.0):
    # median, minimum, maximum and run count of `col` per `by`, scaled for display
    g = d.assign(v=d[col] * scale).groupby(by)["v"]
    return pd.concat([g.median(), g.min(), g.max(), g.count()], axis=1, keys=["med", "min", "max", "n"])


def nest(st, groups):
    # {set: {target: [[med, min, max, n] per group]}}
    out = {}
    for (s, t), g in st.groupby(level=["set", "target"]):
        g = g.droplevel(["set", "target"]).reindex(groups)
        out.setdefault(s, {})[t] = [None if np.isnan(r["n"]) or r["n"] == 0 else [r4(r["med"]), r4(r["min"]), r4(r["max"]), int(r["n"])]
                                    for _, r in g.iterrows()]
    return out


# 7a. markets (notebook 23): 2050 differences and 2026-2050 builds by group
MD = pd.read_parquet(E23 / "mk_delta.parquet").assign(set=lambda d: setkey(d))
FV = pd.read_parquet(E23 / "mk_fleet_value_delta.parquet").assign(set=lambda d: setkey(d))
VF = pd.read_parquet(E23 / "mk_vf_delta.parquet").assign(set=lambda d: setkey(d))
TR = pd.read_csv(E23 / "mk_tran.csv", dtype={"stage": str}).assign(set=lambda d: setkey(d))
assert set(MD["set"]) == set(SETS), sorted(set(MD["set"]))
by = ["set", "target", "group"]
MEAS = {  # page key: (frame, filter, column, scale, unit)
    "gen_share": (MD[(MD["measure"] == "gen_share") & (MD["t"] == Y1)], "delta", 100.0, "percentage points"),
    "cap": (MD[(MD["measure"] == "cap_MW") & (MD["t"] == Y1)], "delta", 1e-3, "GW"),
    "new": (MD[MD["measure"] == "new_MW_2026_2050"], "delta", 1e-3, "GW"),
    "ret": (MD[MD["measure"] == "ret_MW_2026_2050"], "delta", 1e-3, "GW"),
    "rev_mwh": (FV[(FV["measure"] == "rev_total_per_MWh_2025") & (FV["t"] == Y1)], "delta", 1.0, "2025 $/MWh"),
    "vf": (VF[VF["t"] == Y1], "d_vf", 1.0, "value factor"),
}
MSUM, fig_rows = {}, []
for k, (d, col, sc, unit) in MEAS.items():
    st = stats(d, by, col, sc)
    MSUM[k] = nest(st, GROUPS)
    fig_rows.append(st.reset_index().assign(measure=k, unit=unit))
pd.concat(fig_rows).to_csv(OUT / "fig_markets_2050.csv", index=False)
tr50 = TR[TR["t"] == Y1]
MTRAN = {m: {s: {t: [r4(r["med"]), r4(r["min"]), r4(r["max"]), int(r["n"])] for t, r in g.droplevel("set").iterrows()}
             for s, g in stats(tr50[tr50["measure"] == m], ["set", "target"], "delta", 1e-3).groupby(level="set")}
         for m in ["MW", "MW-mi"]}
# check: the stage-1 2050 table of notebook 23 equals the page's stage-1 values
s1t = pd.read_csv(E23 / "mk_stage1_2050.csv").set_index(["quantity", "group"])
for k, q in [("gen_share", "d gen share (pts), 2050"), ("cap", "d capacity (GW), 2050"), ("new", "d new builds 2026-2050 (GW)"),
             ("ret", "d retirements 2026-2050 (GW)")]:
    for t in TARGETS:
        page = pd.Series([v[0] if v else np.nan for v in MSUM[k]["1|reference"][t]], index=GROUPS)
        ref = s1t.loc[q][t].reindex(GROUPS)
        assert np.allclose(page.fillna(0), ref.fillna(0), atol=0.006), (k, t)
print("markets: page values equal mk_stage1_2050.csv to rounding")

# stage-1 annual series (one P50 run per target)
s1 = MD[(MD["stage"] == "1") & (MD["t"] >= 2026)]
YRS = sorted(int(t) for t in s1.loc[s1["measure"] == "gen_share", "t"].unique())
ser = lambda d, m, sc: {t: {g: [r4(v * sc) for v in d[(d["target"] == t) & (d["measure"] == m) & (d["group"] == g)].set_index("t")["delta"].reindex(YRS).values]
                            for g in GROUPS} for t in TARGETS}
fvs = FV[(FV["stage"] == "1") & (FV["measure"] == "rev_total_per_MWh_2025")]
MANN = dict(years=YRS, gen_share=ser(s1, "gen_share", 100.0), cap=ser(s1, "cap_MW", 1e-3),
            rev_mwh={t: {g: [r4(v) for v in fvs[(fvs["target"] == t) & (fvs["group"] == g)].set_index("t")["delta"].reindex(YRS).values]
                         for g in GROUPS} for t in TARGETS},
            tran={t: [r4(v * 1e-3) for v in TR[(TR["stage"] == "1") & (TR["target"] == t) & (TR["measure"] == "MW")].set_index("t")["delta"].reindex(YRS).values]
                  for t in TARGETS})
# value factors of new builds: levels in the six P50 runs and in fbR_none, 2050
vfl = VF[(VF["stage"] == "1") & (VF["t"] == Y1)].set_index(["target", "group"])
MVF = {t: {g: [r4(vfl["vf_run"].get((t, g), np.nan)), r4(vfl["vf_base"].get((t, g), np.nan))] for g in GROUPS} for t in TARGETS}
MLEV = pd.read_parquet(E23 / "mk_levels.parquet")
base50 = MLEV[(MLEV["case"] == "fbR_none") & (MLEV["t"] == Y1)].set_index("group").reindex(GROUPS)
MBASE = dict(cap_GW=[r4(v * 1e-3) for v in base50["cap_MW"]], gen_share=[r4(v * 100) for v in base50["gen_share"]])
# hourly energy value (notebook 23 section 6, amendment A17): stage 1, 2050
HC = pd.read_csv(E23 / "mk_hourly_capture_2035_2050.csv")
HC = HC[HC["t"] == Y1].set_index(["case", "group"])
SEAS, HODS = ["winter", "spring", "summer", "fall"], list(range(3, 25, 3))
def _hc(case, g, col):
    return r4(HC[col].get((case, g), np.nan))
HVF = {t: {g: [_hc(f"smr100_{t}_p50", g, "value_factor"), _hc("fbR_none", g, "value_factor"),
               _hc(f"smr100_{t}_p50", g, "capture_2025"), _hc("fbR_none", g, "capture_2025")] for g in GROUPS} for t in TARGETS}
HP = pd.read_csv(E23 / "mk_hourly_profile_2050.csv")
def _grid(d, col):
    m = d.pivot_table(index="hod", columns="season", values=col).reindex(index=HODS, columns=SEAS)
    return [[r4(v) for v in row] for row in m.values]
HPROF = {t: dict(dprice=_grid(HP[(HP["kind"] == "delta") & (HP["case"] == f"smr100_{t}_p50")], "load_wtd_2025"),
                 smr=_grid(HP[(HP["kind"] == "level") & (HP["case"] == f"smr100_{t}_p50")], "share|SMR (new)"))
         for t in TARGETS}
HBASE = _grid(HP[(HP["kind"] == "level") & (HP["case"] == "fbR_none")], "load_wtd_2025")
chk = HP[(HP["kind"] == "delta") & (HP["case"] == "smr100_eo_p50") & (HP["season"] == "summer") & (HP["hod"] == 18)]["load_wtd_2025"].iloc[0]
assert abs(HPROF["eo"]["dprice"][HODS.index(18)][SEAS.index("summer")] - chk) < 1e-3
print("hourly: page values equal mk_hourly_*_2050 files to rounding")
hourly = dict(seasons=SEAS, hods=HODS, vf=HVF, prof=HPROF, base=HBASE)
markets = dict(groups=GROUPS, sets=[dict(id=s, label=SET_LABEL[s]) for s in SETS], summary=MSUM, tran=MTRAN,
               annual=MANN, vf=MVF, base2050=MBASE, hourly=hourly,
               counts=MD[MD["measure"] == "gen_share"].groupby(["set", "target"])["case"].nunique().unstack().reindex(SETS).fillna(0).astype(int).to_dict("index"))

# 7b. census divisions (notebook 26)
CP = pd.read_parquet(E26 / "cd_pv.parquet")
CP["set"] = setkey(CP)
CP["a"] = np.where(CP["actor"] == "environment", CP["entry"], CP["actor"])
DIVS = sorted(CP["cendiv"].unique())
own = CP[CP["actor"].str.startswith("owners: ")].assign(a="owners (all groups)")
CPA = pd.concat([CP, own])
CD_ACTORS = (["electricity consumers", "owners (all groups)"] + [f"owners: {g}" for g in GROUPS if g != "Canadian imports"]
             + ["foreign sellers: Canadian imports", "system", "readout"] + [f"health: {v}" for v in HV])
assert set(CD_ACTORS) == set(CPA["a"].unique()), set(CPA["a"].unique()) ^ set(CD_ACTORS)
VK = [f"{hz}|{r}" for hz in ("A", "B") for r in RATE_KEYS]
CPA["vk"] = CPA["horizon"] + "|" + CPA["rate"]
run_v = CPA.groupby(["set", "target", "case", "vk", "a", "cendiv"])["pv"].sum()
cd_s = run_v.groupby(level=["set", "target", "vk", "a", "cendiv"]).agg(["median", "min", "max", "count"])
CDS = {}
for (s, t, vk), g in cd_s.groupby(level=["set", "target", "vk"]):
    g = g.droplevel(["set", "target", "vk"])
    CDS.setdefault(s, {}).setdefault(t, {})[vk] = [[([r4(g.loc[(a, dv), "median"]), r4(g.loc[(a, dv), "min"]), r4(g.loc[(a, dv), "max"])]
                                                     if (a, dv) in g.index else None) for dv in DIVS] for a in CD_ACTORS]
cd_s.reset_index().to_csv(OUT / "fig_cendiv_pv.csv", index=False)
# check: stage-1 values equal notebook 26's stage-1 table (horizon A, 2.0%)
cs1 = pd.read_csv(E26 / "cd_stage1.csv").set_index(["actor", "cendiv"])
for a in ["electricity consumers", "owners (all groups)", "system", "readout", "health: AP2 ACS"]:
    for t in TARGETS:
        page = pd.Series([v[0] for v in CDS["1|reference"][t]["A|2.0%"][CD_ACTORS.index(a)]], index=DIVS)
        assert np.allclose(page, cs1.loc[a][t].reindex(DIVS), atol=6e-4), (a, t)
print("census divisions: page values equal cd_stage1.csv to rounding")
# division totals against the national ledger (the notebook 26 checks)
cchk = pd.read_csv(E26 / "cd_check.csv")
# 2050 annual emissions by division (process CO2 in Mt per year; NOx and SO2 in kt per year)
CE = pd.read_parquet(E26 / "cd_emissions.parquet")
CE = CE[(CE["t"] == Y1) & (CE["etype"] == "process") & CE["eall"].isin(["CO2", "NOX", "SO2"])].assign(set=lambda d: setkey(d))
EMSC = {"CO2": 1e-6, "NOX": 1e-3, "SO2": 1e-3}
CE["v"] = CE["delta_t"] * CE["eall"].map(EMSC)
ce_s = CE.groupby(["set", "target", "eall", "cendiv", "case"])["v"].sum().groupby(level=["set", "target", "eall", "cendiv"]).agg(["median", "min", "max"])
CDE = {}
for (s, t, gs), g in ce_s.groupby(level=["set", "target", "eall"]):
    g = g.droplevel(["set", "target", "eall"]).reindex(DIVS)
    CDE.setdefault(s, {}).setdefault(t, {})[gs] = [[r4(r["median"]), r4(r["min"]), r4(r["max"])] for _, r in g.iterrows()]
nat_co2 = CE[(CE["set"] == "1|reference") & (CE["eall"] == "CO2")].groupby("target")["v"].sum()
print("stage-1 Δ process CO2 2050, summed over divisions (Mt/yr):", nat_co2.reindex(TARGETS).round(1).to_dict())
cendiv = dict(divisions=DIVS, actors=CD_ACTORS, vkeys=VK, sets=[dict(id=s, label=SET_LABEL[s]) for s in SETS],
              pv=CDS, emis=CDE, check_max_rel=float(cchk["rel"].max()), check_runs=int(cchk["case"].nunique()),
              counts=CP.groupby(["set", "target"])["case"].nunique().unstack().reindex(SETS).fillna(0).astype(int).to_dict("index"))
for name, obj in (("markets", markets), ("cendiv", cendiv)):
    s = json.dumps(clean(obj), separators=(",", ":"), allow_nan=False)
    (PAGE / "data" / f"{name}.json").write_text(s, encoding="utf-8")
    print(f"{name}.json: {len(s) / 1e6:.2f} MB")

markets: page values equal mk_stage1_2050.csv to rounding


hourly: page values equal mk_hourly_*_2050 files to rounding


census divisions: page values equal cd_stage1.csv to rounding


stage-1 Δ process CO2 2050, summed over divisions (Mt/yr): {'eia': -17.4, 'aj': -22.9, 'iaea': -58.0, 'mck': -99.1, 'cop28': -271.9, 'eo': -459.6}


markets.json: 0.12 MB
cendiv.json: 3.22 MB


## 8. Retail rates (notebook 27, amendment A21)

Ethan confirmed A21 on 2026-11-13 (Decisions, item 18), so the retail readouts of notebook 27 go on the page.
They come from the patched retail module (O33, O34). This cell writes `data/retail.json`, which the retail page
loads on demand: the national rate of `fbR_none` and of each P50 run, Δrate by year (national and by census
division), the split of Δrate into the module's categories, and the PV of the consumers' retail gain beside
notebook 21's wholesale gain. All rates are 2025 ¢/kWh; years 2010–2050 as the module writes them. The check: the
national Δrate rebuilt from the rates equals the sum of the category split, and the module's own national rate
equals the rebuild (`rr_check_national.csv`).

In [11]:
E27 = paths.exports("27")
RR = pd.read_csv(E27 / "rr_rates.csv")
RC = pd.read_csv(E27 / "rr_delta_components.csv")
RD = pd.read_csv(E27 / "rr_delta_cendiv.csv")
RPV = pd.read_csv(E27 / "rr_pv.csv")
RCK = pd.read_csv(E27 / "rr_check_national.csv")
r4 = lambda x: float(round(float(x), 4))
nat = RR[RR["scope"] == "national"].pivot_table(index="t", columns="case", values="rate_c2025_per_kWh").sort_index()
YR = [int(t) for t in nat.index]
case_of = {t: f"smr100_{t}_p50" for t in TARGETS}
dnat = {t: nat[case_of[t]] - nat["fbR_none"] for t in TARGETS}
GROUPS_R = list(RC.groupby("group", sort=False).size().index)
comp = {t: {g: [r4(v) for v in RC[(RC["target"] == t) & (RC["group"] == g)].set_index("t")["delta_c2025_per_kWh"].reindex(YR).fillna(0.0)]
            for g in GROUPS_R} for t in TARGETS}
for t in TARGETS:                                     # the category split sums to the national Δrate
    s = np.sum([comp[t][g] for g in GROUPS_R], axis=0)
    assert np.abs(s - dnat[t].values).max() < 5e-4, (t, np.abs(s - dnat[t].values).max())
DIVS_R = sorted(RD["cendiv"].unique())
cd = {t: {d: [r4(v) for v in RD[(RD["target"] == t) & (RD["cendiv"] == d)].set_index("t")["delta_c2025_per_kWh"].reindex(YR)]
          for d in DIVS_R} for t in TARGETS}
cdr = RR[RR["scope"] == "cendiv"].pivot_table(index="t", columns=["region", "case"], values="rate_c2025_per_kWh")
cd_base = {d: [r4(v) for v in cdr[(d, "fbR_none")].reindex(YR)] for d in DIVS_R}
pv = {t: {r: dict(retail=r3(g["retail_gain_pv_2025B"].iloc[0]), wholesale=r3(g["wholesale_gain_pv_2025B"].iloc[0]))
          for r, g in RPV[RPV["target"] == t].groupby("rate")} for t in TARGETS}
retail = dict(years=YR, base=[r4(v) for v in nat["fbR_none"]], rate={t: [r4(v) for v in nat[case_of[t]]] for t in TARGETS},
              delta={t: [r4(v) for v in dnat[t]] for t in TARGETS}, groups=GROUPS_R, comp=comp, divisions=DIVS_R,
              cd=cd, cd_base=cd_base, pv=pv, check_max=float(RCK["diff"].abs().max()), check_n=int(len(RCK)))
s = json.dumps(clean(retail), separators=(",", ":"), allow_nan=False)
(PAGE / "data" / "retail.json").write_text(s, encoding="utf-8")
print(f"retail.json: {len(s) / 1e6:.2f} MB; module vs rebuild max |diff| {retail['check_max']:.1e} over {retail['check_n']} case-years")
print("2050 national Δrate:", {t: retail["delta"][t][-1] for t in TARGETS})
print("PV at 2.0%:", {t: pv[t]["2.0%"] for t in TARGETS})


retail.json: 0.04 MB; module vs rebuild max |diff| 1.2e-14 over 287 case-years
2050 national Δrate: {'eia': 0.0648, 'aj': 0.1916, 'iaea': 0.2103, 'mck': 0.3424, 'cop28': 0.5937, 'eo': 1.0184}
PV at 2.0%: {'eia': {'retail': -48.672, 'wholesale': 32.699}, 'aj': {'retail': -60.056, 'wholesale': 51.136}, 'iaea': {'retail': -147.555, 'wholesale': 133.359}, 'mck': {'retail': -180.154, 'wholesale': 162.563}, 'cop28': {'retail': -298.283, 'wholesale': 353.369}, 'eo': {'retail': -453.038, 'wholesale': 585.897}}


## 8b. RUN-1 to 2055 (notebook 29, item 21)

Ethan chose descriptive tables for RUN-1 on 2026-11-16 (Decisions, item 21): report the SMR value, the required support
and their change after 2050, with no claim. This cell writes `data/run1.json` from notebook 29's exports: every
readout by target, run (RUN-1, `_ext`) and solve year 2041–2055. The check: the two runs agree in 2050 (O38).

In [12]:
R29 = pd.read_csv(paths.exports("29") / "run1_readouts.csv")
Q29 = [c for c in R29.columns if c not in ("target", "run", "t")]
R29 = R29[R29["t"] >= 2041]
Y29 = sorted(R29["t"].unique().astype(int).tolist())
run1 = dict(years=Y29, quantities=Q29,
            data={t: {w: {q: [None if pd.isna(v) else r4(v) for v in g.set_index("t")[q].reindex(Y29)] for q in Q29}
                      for w, g in R29[R29["target"] == t].groupby("run")} for t in TARGETS})
for t in TARGETS:
    for q in ["SMR fleet, GW", "required support, 2025 $/kW", "fleet revenue, 2025 $/kW-yr"]:
        a, b = run1["data"][t]["RUN-1"][q][Y29.index(2050)], run1["data"][t]["_ext"][q][Y29.index(2050)]
        assert abs(a - b) <= 1e-3 * max(abs(b), 1.0) + 0.2, (t, q, a, b)
s = json.dumps(clean(run1), separators=(",", ":"), allow_nan=False)
(PAGE / "data" / "run1.json").write_text(s, encoding="utf-8")
print(f"run1.json: {len(s) / 1e3:.0f} kB; years {Y29}")

run1.json: 10 kB; years [2041, 2044, 2047, 2050, 2053, 2055]
